# Notebook 01 — Data Exploration

Explores the MNIST digit dataset and any custom operator images,
shows class distributions, and visualises sample images from each class.

**Run order:** run this notebook before training to verify your dataset is healthy.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import config

## 1. Load MNIST

In [ ]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f'Train samples : {len(x_train):,}')
print(f'Test  samples : {len(x_test):,}')
print(f'Image shape   : {x_train.shape[1:]}')

## 2. MNIST class distribution

In [ ]:
import pandas as pd

counts = pd.Series(y_train).value_counts().sort_index()
counts.index = [str(i) for i in counts.index]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
ax.set_title('MNIST training set — samples per digit class')
ax.set_xlabel('Digit'); ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('../output/reports/01_mnist_distribution.png', dpi=120)
plt.show()
print('Saved distribution chart.')

## 3. Sample images — all MNIST digit classes

In [ ]:
n_samples = 8
fig, axes = plt.subplots(10, n_samples, figsize=(n_samples * 1.2, 13))
fig.suptitle('MNIST — 8 samples per digit class', fontsize=12)

for digit in range(10):
    idxs = np.where(y_train == digit)[0][:n_samples]
    for j, idx in enumerate(idxs):
        axes[digit, j].imshow(x_train[idx], cmap='gray')
        axes[digit, j].axis('off')
        if j == 0:
            axes[digit, j].set_ylabel(str(digit), fontsize=10, rotation=0, labelpad=15, va='center')

plt.tight_layout()
plt.savefig('../output/reports/01_mnist_samples.png', dpi=120)
plt.show()

## 4. Custom operator images

In [ ]:
import cv2
from src.data_preparation import DataPreparator

operator_summary = []
for class_idx, symbol in config.CLASS_MAP.items():
    if class_idx < 10:
        continue
    folder_name = config.SYMBOL_FOLDER_MAP.get(class_idx)
    if folder_name is None:
        continue
    folder = config.SYMBOLS_DIR / folder_name
    images = list(folder.glob('*.png')) + list(folder.glob('*.jpg'))
    operator_summary.append({'class': class_idx, 'symbol': symbol,
                              'folder': folder_name, 'n_images': len(images)})
    print(f"  class {class_idx:2d}  '{symbol}'  ({folder_name})  — {len(images)} images")

if not any(r['n_images'] for r in operator_summary):
    print('\n  No custom operator images found.')
    print('  Add hand-drawn images to data/symbols/{plus,minus,...}/ before training.')

## 5. Pixel statistics on MNIST

In [ ]:
x_norm = x_train.astype(np.float32) / 255.0
print(f'Min  : {x_norm.min():.4f}')
print(f'Max  : {x_norm.max():.4f}')
print(f'Mean : {x_norm.mean():.4f}')
print(f'Std  : {x_norm.std():.4f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(x_norm.flatten(), bins=50, color='cornflowerblue', edgecolor='none')
ax.set_title('Pixel value distribution (normalised MNIST training set)')
ax.set_xlabel('Pixel value'); ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('../output/reports/01_pixel_hist.png', dpi=120)
plt.show()

## 6. DataPreparator — full dataset build (optional)

Uncomment the cell below to build the full 16-class dataset (requires operator images).

In [ ]:
# from src.data_preparation import DataPreparator
# dp = DataPreparator()
# X_train, X_val, X_test, y_train_full, y_val, y_test_full = dp.prepare()
# print('Dataset shapes:')
# print(f'  Train : {X_train.shape}, labels {y_train_full.shape}')
# print(f'  Val   : {X_val.shape}')
# print(f'  Test  : {X_test.shape}')